# Project 04 - gray_image 숫자 예측 결과 확인

위에서부터 순서대로 실행하면 `gray_image.png`의 예측 숫자, 예측 확률, MNIST 테스트 정확도를 확인할 수 있습니다.

패키지 오류가 나면 바로 아래 설치 셀의 주석을 해제하고 한 번 실행하세요.

In [ ]:
# 패키지 오류가 날 때만 아래 줄의 #을 지우고 실행하세요.
# %pip install tensorflow opencv-python matplotlib numpy

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

In [ ]:
# 실행 위치가 프로젝트 루트이든 camera 폴더이든 gray_image.png를 찾습니다.
candidate_paths = [
    Path.cwd() / "gray_image.png",
    Path.cwd() / "camera" / "gray_image.png",
]

image_path = next((path for path in candidate_paths if path.exists()), None)

if image_path is None:
    raise FileNotFoundError("gray_image.png를 찾을 수 없습니다. project_02에서 이미지를 먼저 저장하세요.")

print("사용할 이미지:", image_path)

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train_norm = x_train.reshape(60000, 28 * 28).astype("float32") / 255.0
x_test_norm = x_test.reshape(10000, 28 * 28).astype("float32") / 255.0

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(28 * 28,)),
    tf.keras.layers.Dense(units=512, activation="relu", name="Hidden1"),
    tf.keras.layers.Dense(units=256, activation="relu", name="Hidden2"),
    tf.keras.layers.Dense(units=10, activation="softmax", name="OutputLayer"),
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model.fit(
    x_train_norm,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
)

test_loss, test_accuracy = model.evaluate(x_test_norm, y_test, verbose=0)

print(f"MNIST 테스트 정확도: {test_accuracy * 100:.2f}%")
print(f"MNIST 테스트 손실: {test_loss:.4f}")

In [ ]:
gray_image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

if gray_image is None:
    raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {image_path}")

blur = cv2.GaussianBlur(gray_image, (5, 5), 0)
_, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# MNIST 입력은 검은 배경 + 밝은 숫자에 가깝게 맞춥니다.
border_pixels = np.concatenate([binary[0, :], binary[-1, :], binary[:, 0], binary[:, -1]])
mnist_style = 255 - binary if border_pixels.mean() > 127 else binary.copy()

resize_img = cv2.resize(mnist_style, (28, 28), interpolation=cv2.INTER_AREA)
input_data = (resize_img.astype("float32") / 255.0).reshape(1, 28 * 28)

result = model.predict(input_data, verbose=0)[0]
predicted_digit = int(np.argmax(result))
confidence = float(result[predicted_digit])

print("=" * 40)
print(f"예측 숫자: {predicted_digit}")
print(f"예측 확률: {confidence * 100:.2f}%")
print(f"MNIST 테스트 정확도: {test_accuracy * 100:.2f}%")
print("=" * 40)

for digit, prob in enumerate(result):
    print(f"{digit}: {prob * 100:6.2f}%")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

axes[0].imshow(gray_image, cmap="gray")
axes[0].set_title("gray_image")

axes[1].imshow(mnist_style, cmap="gray")
axes[1].set_title("MNIST style")

axes[2].imshow(resize_img, cmap="gray")
axes[2].set_title("28x28")

axes[3].bar(range(10), result)
axes[3].set_xticks(range(10))
axes[3].set_ylim(0, 1)
axes[3].set_title(f"Prediction: {predicted_digit} ({confidence * 100:.2f}%)")

for ax in axes[:3]:
    ax.axis("off")

plt.tight_layout()
plt.show()